# Fig. S4 — distribution of polygon sides at $t = 0$ and $t = 15000$

In [ ]:
from vertex_lite.custom import load_data_v3 as load_data
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import pyprojroot.here as here

In [ ]:
def filter_data(time, pcts=[0.05, 0.25], noises=[0.2, 0.7], local=False, cluster=False, summary=False):
    rpath = here("results/resultados_cluster") if cluster else here("results/resultados_random")

    dfs = []
    for noise in noises:
        for pct in pcts:
            dfs.append(
                load_data(
                    f"{rpath}/noise_{noise}/pct_{pct}/sim_*/cells.parquet",
                    columns=["time", "n_lados", "neighbor_of_mutant", "cell_id", "alive"],
                    time_filter=time
                )
            )

    df = pd.concat(dfs, ignore_index=True)

    if df.empty:
        print(f"No data found for time={time}, pcts={pcts}, noises={noises}, local={local}, cluster={cluster}")
        return pd.DataFrame()

    df = df[df["alive"] == 1].copy()

    if local:
        df = df[df["neighbor_of_mutant"] == True]
    else:
        df = df.drop(columns=["neighbor_of_mutant"])

    df = (
        df[df["n_lados"] >= 3]
        .groupby(["time", "mesh_type", "pct_mutant", "sim_id", "n_lados"])
        .size()
        .reset_index(name="n_cells")
    )

    # absent polygon classes count as 0 cells in every simulation
    # (only classes observed in at least one simulation of each condition are kept)
    keys = ["time", "mesh_type", "pct_mutant", "sim_id"]
    df = (
        df.set_index(keys + ["n_lados"])["n_cells"]
        .unstack(fill_value=0)
        .reset_index()
        .melt(id_vars=keys, var_name="n_lados", value_name="n_cells")
    )
    df = df[df.groupby(["time", "mesh_type", "pct_mutant", "n_lados"])["n_cells"].transform("sum") > 0]

    df["total_cells"] = (
        df.groupby(["time", "mesh_type", "pct_mutant", "sim_id"])["n_cells"]
        .transform("sum")
    )
    df["frac_cells"] = df["n_cells"] / df["total_cells"]

    stats = (
        df.groupby(["time", "mesh_type", "pct_mutant", "n_lados"])["frac_cells"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    stats["sem"] = stats["std"] / np.sqrt(stats["count"])

    if summary:
        return stats
    else:
        return df

In [ ]:
def plotting(pct, noise,local, cluster):
    tend = 14980  # last saved time step (t = 15000)

    raw = pd.concat(
        [
            filter_data(time=0,pcts=pct,noises=noise, local=local, cluster=cluster, summary=False).assign(label="t0"),
            filter_data(time=tend,pcts=pct,noises=noise,local=local, cluster=cluster, summary=False).assign(label="tend"),
        ],
        ignore_index=True
    )

    stats = pd.concat(
        [
            filter_data(time=0,pcts=pct,noises=noise, local=local, cluster=cluster, summary=True).assign(label="t0"),
            filter_data(time=tend,pcts=pct,noises=noise, local=local, cluster=cluster, summary=True).assign(label="tend"),
        ],
        ignore_index=True
    )

    sns.set_theme(style="white")
    g = sns.FacetGrid(
        raw,
        row="mesh_type",
        col="pct_mutant",
        height=3,
        aspect=1.4,
        sharex=True,
        sharey=True,
        margin_titles=True
    )

    def draw_panel(data, **kwargs):
        ax = plt.gca()

        mesh = data["mesh_type"].iloc[0]
        pct = data["pct_mutant"].iloc[0]

        raw_sub = raw[(raw["mesh_type"] == mesh) & (raw["pct_mutant"] == pct)]
        stats_sub = stats[(stats["mesh_type"] == mesh) & (stats["pct_mutant"] == pct)]

        # individual simulations
        for label, offset in [("t0", -0.12), ("tend", 0.12)]:
            d = raw_sub[raw_sub["label"] == label]
            x = d["n_lados"].astype(float).to_numpy() + offset
            y = d["frac_cells"].to_numpy()

            ax.scatter(
                x, y,
                s=18,
                alpha=0.35,
                color="gray"
            )

        # mean ± sem
        colors = {"t0": "#2166ac", "tend": "#b2182b"}
        for label, offset in [("t0", -0.12), ("tend", 0.12)]:
            d = stats_sub[stats_sub["label"] == label]
            x = d["n_lados"].astype(float).to_numpy() + offset
            y = d["mean"].to_numpy()
            yerr = d["sem"].to_numpy()

            ax.errorbar(
                x, y, yerr=yerr,
                fmt="_",
                linestyle="none",
                color=colors[label],
                capsize=3,
                markersize=18,
                elinewidth=1.2
            )

        ax.grid(False)

    g.map_dataframe(draw_panel)
    g.set_titles(
    row_template="D = {row_name}",
    col_template="% Mutants = {col_name}"
)
    nrows, ncols = g.axes.shape
    scope = "Local" if local else "Global"

    for i, row_axes in enumerate(g.axes):
        for j, ax in enumerate(row_axes):
            ax.set_ylim([-0.05, 0.85])
            ax.set_xlim(2.5, 11.5)
            ax.set_xticks([3,4,5,6,7,8,9,10,11])

            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            ax.tick_params(
                axis="both", which="both", direction="out",
                length=4, labelsize=13, width=1
            )

            ax.tick_params(axis="x", rotation=45)

            ax.xaxis.set_ticks_position("bottom")
            ax.yaxis.set_ticks_position("left")

            # x labels only on the bottom row
            if i == nrows - 1:
                ax.set_xlabel("Number of Neighbours")
                ax.tick_params(axis="x", labelbottom=True)
            else:
                ax.set_xlabel("")
                ax.tick_params(axis="x", labelbottom=False)

            # y labels only on the left column
            if j == 0:
                ax.set_ylabel("Fraction of cells")
                ax.tick_params(axis="y", labelleft=True)
            else:
                ax.set_ylabel("")
                ax.tick_params(axis="y", labelleft=False)

    savedir = here("results/histograms")
    os.makedirs(savedir, exist_ok=True)
    filename = f"{savedir}/Hist_{'Cluster' if cluster else 'uniform'}_{scope}_pct{pct}_noise{noise}.svg"
    plt.savefig(filename, format="svg", bbox_inches="tight")
    plt.close()

In [ ]:
pcts = [0.05, 0.25]
noises = [0.2, 0.7]
for local in [True, False]:
    for cluster in [True, False]:
        plotting(pct=pcts, noise=noises, local=local, cluster=cluster)